
# cap10_foldlocal_joint_target_contract_v1

Purpose: implement the next mechanism from `cap10_foldlocal_multi_objective_readiness_v1` by creating a fold-local joint target contract that combines sparse-release rebound lift and drawdown-control rows without evaluation leakage.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison: `cap10_foldlocal_multi_objective_readiness_v1`.
- Role: `diagnose`.
- Changed layer: `model_selection`, `training_objective`, `src helper`.
- Target: prepare non-leaking labels for a later BC/PPO screen; this notebook is not a backtest and cannot promote a model.
- Acceptance gate: fold-local scope, `eval_overlap_rows == 0`, positive rebound lift under risk-priority target semantics, downtrend-control rows, healthy action distribution, and anti-overfit stop rules before any PPO extension.
- External basis: Qlib artifact discipline, PBO/CSCV, Deflated Sharpe, purged/embargo validation, conservative offline RL, and risk-sensitive RL.

Closest historical success: `sparse_rebound_release_target_revision_v1` passed fold-local scope/lift/readiness gates.

Closest historical failure: `sparse_release_bc_ppo_screen_v1` improved `wf_2022` relative return but selected a checkpoint with `37.28%` MaxDD, so checkpoint selection must reject drawdown drift.

Why this is not a duplicate: this run uses a new tested `build_fold_local_joint_target_contract()` helper to produce one target table that contains both sparse-release lift and drawdown-control rows in each fold-local calendar.


In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    build_fold_local_joint_target_contract,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)

EXPERIMENT = "cap10_foldlocal_joint_target_contract_v1"
ROOT = Path.cwd()
CARD_ROOT = ROOT / "runtime" / "model_selection" / "card"
OUT_DIR = CARD_ROOT / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0


In [2]:

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text()) if path.exists() else {}


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def derive_folds(targets: pd.DataFrame) -> list[dict[str, str]]:
    required = ["fold", "train_start", "train_end", "eval_start", "eval_end"]
    missing = sorted(set(required) - set(targets.columns))
    if missing:
        raise ValueError(f"source targets missing fold metadata: {missing}")
    folds = []
    for fold, group in targets.groupby("fold", sort=True):
        unique = group[required].drop_duplicates()
        if len(unique) != 1:
            raise ValueError(f"fold {fold} has inconsistent boundary rows")
        row = unique.iloc[0]
        folds.append(
            {
                "name": str(fold),
                "train_start": str(row["train_start"]),
                "train_end": str(row["train_end"]),
                "eval_start": str(row["eval_start"]),
                "eval_end": str(row["eval_end"]),
            }
        )
    return folds


def target_action_distribution(targets: pd.DataFrame) -> pd.DataFrame:
    if targets.empty:
        return pd.DataFrame()
    frame = targets.copy()
    for col in ["teacher_position", "rebound_target_position", "risk_target_position"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0.0)
    for col in ["is_rebound_cluster", "is_downtrend_state"]:
        frame[col] = frame[col].fillna(False).astype(bool)
    target_position = np.where(
        frame["is_downtrend_state"],
        frame["risk_target_position"],
        np.where(frame["is_rebound_cluster"], frame["rebound_target_position"], frame["teacher_position"]),
    )
    frame["target_action"] = np.round(np.clip(target_position, 0.0, 1.0) / 0.05).astype(int)
    rows = []
    for profile, group in frame.groupby("fold", sort=True):
        counts = group["target_action"].value_counts(normalize=True)
        rows.append(
            {
                "profile": str(profile),
                "samples": int(len(group)),
                "unique_actions": int(group["target_action"].nunique()),
                "dominant_action": int(counts.index[0]),
                "dominant_action_share": float(counts.iloc[0]),
                "action_entropy": float(-(counts * np.log(counts + 1e-12)).sum()),
                "rebound_rows": int(group["is_rebound_cluster"].sum()),
                "downtrend_rows": int(group["is_downtrend_state"].sum()),
            }
        )
    all_counts = frame["target_action"].value_counts(normalize=True)
    rows.append(
        {
            "profile": "all_folds_joint_contract",
            "samples": int(len(frame)),
            "unique_actions": int(frame["target_action"].nunique()),
            "dominant_action": int(all_counts.index[0]),
            "dominant_action_share": float(all_counts.iloc[0]),
            "action_entropy": float(-(all_counts * np.log(all_counts + 1e-12)).sum()),
            "rebound_rows": int(frame["is_rebound_cluster"].sum()),
            "downtrend_rows": int(frame["is_downtrend_state"].sum()),
        }
    )
    return pd.DataFrame(rows)


In [3]:

readiness = load_json(CARD_ROOT / "cap10_foldlocal_multi_objective_readiness_v1" / "selection.json")
release_selection = load_json(CARD_ROOT / "sparse_rebound_release_target_revision_v1" / "selection.json")
drawdown_selection = load_json(CARD_ROOT / "sparse_release_drawdown_control_foldlocal_bc_warmstart_v1" / "selection.json")
ppo_selection = load_json(CARD_ROOT / "sparse_release_bc_ppo_screen_v1" / "selection.json")
dd_ppo_selection = load_json(CARD_ROOT / "sparse_release_drawdown_control_bc_ppo_screen_v1" / "selection.json")

release_targets = read_csv(CARD_ROOT / "sparse_rebound_release_target_revision_v1" / "sparse_release_revised_targets.csv")
drawdown_targets = read_csv(CARD_ROOT / "sparse_release_drawdown_control_foldlocal_bc_warmstart_v1" / "selected_foldlocal_targets.csv")
folds = derive_folds(release_targets)

joint_targets = build_fold_local_joint_target_contract(
    release_targets,
    drawdown_targets,
    folds=folds,
)

joint_summary = joint_targets.groupby("fold", sort=True).agg(
    rows=("datetime", "size"),
    start=("datetime", "min"),
    end=("datetime", "max"),
    rebound_rows=("is_rebound_cluster", "sum"),
    downtrend_rows=("is_downtrend_state", "sum"),
    release_gate_rows=("release_gate_state", "sum"),
    drawdown_control_rows=("drawdown_control_state", "sum"),
    source_component_count_max=("source_component_count", "max"),
).reset_index()

joint_summary


,fold,rows,start,end,rebound_rows,downtrend_rows,release_gate_rows,drawdown_control_rows,source_component_count_max
0,wf_2022_train,505,2020-01-02,2021-12-31,66,234,40,49,2
1,wf_2023_2024_train,756,2020-01-02,2022-12-30,66,478,40,49,2


In [4]:

scope_audit = audit_state_objective_target_scope(joint_targets, folds=folds)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
lift_audit = audit_state_objective_target_lift(
    joint_targets,
    group_cols=("fold",),
    baseline_risk_position_col="risk_target_position_original",
)
lift_summary = summarize_state_objective_target_lift_audit(lift_audit)
action_distribution = target_action_distribution(joint_targets)
bc_readiness_scores = score_state_objective_bc_readiness(
    action_distribution.rename(columns={"profile": "profile"}),
    target_scope="fold_local",
)
bc_readiness_summary = summarize_state_objective_bc_readiness(
    bc_readiness_scores[bc_readiness_scores["profile"] == "all_folds_joint_contract"]
)

display(pd.DataFrame([scope_summary]))
display(lift_audit)
display(bc_readiness_scores)


,decision,training_allowed,leakage_gate,audited_groups,blocked_groups,eval_overlap_rows,outside_train_eval_rows,scope_issues,top_scope_issue,next_mechanism
0,state_objective_targets_training_ready,True,True,2,0,0,0,[],,run_fold_local_state_objective_training_or_bc


,fold,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,wf_2022_train,505,66,234,57,40,57,0.226059,0.34186,0.556976,0.556976,True,True,risk,1.000000e-09
1,wf_2023_2024_train,756,66,478,57,40,57,0.226059,0.34186,0.556976,0.556976,True,True,risk,1.000000e-09


,profile,samples,unique_actions,dominant_action,dominant_action_share,action_entropy,rebound_rows,downtrend_rows,sample_gate,label_distribution_gate,fold_local_scope_gate,bc_readiness_gate,bc_readiness_loss,target_scope
0,wf_2022_train,505,17,4,0.211881,2.488052,66,234,True,True,True,True,0.0,fold_local
1,all_folds_joint_contract,1261,18,4,0.221253,2.458781,132,712,True,True,True,True,0.0,fold_local
2,wf_2023_2024_train,756,18,4,0.227513,2.383438,66,478,True,True,True,True,0.0,fold_local


In [5]:

previous_training_failures = readiness.get("selected_training_failures", [])
coverage_gate = {str(row["name"]): True for row in folds}
wf_2020_coverage = any("2020" in str(row["name"]) and str(row["name"]) not in {"wf_2022_train", "wf_2023_2024_train"} for row in folds)
target_gate = bool(
    scope_summary.get("training_allowed", False)
    and scope_summary.get("eval_overlap_rows", 1) == 0
    and scope_summary.get("outside_train_eval_rows", 1) == 0
    and lift_summary.get("training_allowed", False)
    and bc_readiness_summary.get("selected_bc_readiness_gate", False)
)

anti_overfit_stop_rules = pd.DataFrame(
    [
        {
            "rule": "absolute_fold_gate",
            "condition": "every selected fold must have relative_return >= 30 and max_dd <= 15",
            "required_before": "retaining any PPO checkpoint",
        },
        {
            "rule": "drawdown_drift_reject",
            "condition": "reject selected PPO if selected_vs_bc_drawdown_delta > 2pp and absolute fold gate is false",
            "required_before": "checkpoint selection",
        },
        {
            "rule": "bc_screen_before_ppo_extension",
            "condition": "run a BC-only screen on release/downtrend slices before longer PPO",
            "required_before": "PPO extension",
        },
        {
            "rule": "wf_2020_coverage_before_promotion",
            "condition": "promotion review must include non-leaking wf_2020 target/replay coverage or explicitly block promotion",
            "required_before": "promotion review",
        },
        {
            "rule": "target_contract_reaudit",
            "condition": "rerun scope, lift, and BC readiness audits after any target filtering or sampling change",
            "required_before": "training rerun",
        },
    ]
)

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "cap10_foldlocal_multi_objective_readiness_v1",
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_objective", "src_helper"],
    "decision": "foldlocal_joint_target_contract_ready_for_bc_screen_no_promotion" if target_gate else "foldlocal_joint_target_contract_blocked",
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "bc_screen_allowed_next": bool(target_gate),
    "target_gate": bool(target_gate),
    "target_scope_gate": bool(scope_summary.get("training_allowed", False)),
    "eval_overlap_rows": int(scope_summary.get("eval_overlap_rows", -1)),
    "outside_train_eval_rows": int(scope_summary.get("outside_train_eval_rows", -1)),
    "target_lift_gate": bool(lift_summary.get("training_allowed", False)),
    "current_positive_rebound_lift_rows": int(lift_summary.get("current_positive_rebound_lift_rows", 0)),
    "candidate_release_positive_lift_rows": int(lift_summary.get("candidate_release_positive_lift_rows", 0)),
    "rebound_downtrend_rows": int(lift_summary.get("rebound_downtrend_rows", 0)),
    "bc_readiness_gate": bool(bc_readiness_summary.get("selected_bc_readiness_gate", False)),
    "selected_samples": int(bc_readiness_summary.get("selected_samples", 0)),
    "selected_unique_actions": int(bc_readiness_summary.get("selected_unique_actions", 0)),
    "selected_dominant_action_share": float(bc_readiness_summary.get("selected_dominant_action_share", 1.0)),
    "folds": [row["name"] for row in folds],
    "wf_2020_coverage": bool(wf_2020_coverage),
    "previous_training_failures": previous_training_failures,
    "next_mechanism": "run_joint_contract_bc_screen_with_drawdown_drift_stop_rules",
    "promotion_blocker": "Target contract readiness is not a backtest; previous PPO screens showed drawdown drift, and promotion still requires fixed-split plus wf_2020/wf_2022/wf_2023_2024 replay gates.",
    "acceptance_gate_for_next_run": {
        "bc_slice_screen_required": True,
        "checkpoint_overfit_stop_rules_required": True,
        "relative_return_min_pp": TARGET_RELATIVE_RETURN,
        "max_dd_max_pct": TARGET_MAX_DD,
        "walkforward_required": True,
        "fixed_split_required": True,
        "wf_2020_coverage_required_before_promotion": True,
        "no_runtime_promotion_from_target_readiness_only": True,
    },
    "external_sources": sorted({
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://doi.org/10.21314/JCF.2016.322",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678",
        "https://arxiv.org/abs/2006.04779",
        "https://arxiv.org/abs/1506.02188",
    }),
    "sidecars": [
        "joint_target_contract.csv",
        "joint_target_summary.csv",
        "target_scope_audit.csv",
        "target_lift_audit.csv",
        "action_distribution.csv",
        "bc_readiness_scores.csv",
        "anti_overfit_stop_rules.csv",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    **selection,
    "source_target_rows": {
        "sparse_release_revised_targets": int(len(release_targets)),
        "drawdown_control_targets": int(len(drawdown_targets)),
        "joint_contract": int(len(joint_targets)),
    },
    "source_decisions": {
        "readiness": readiness.get("decision", ""),
        "release_revision": release_selection.get("decision", ""),
        "drawdown_warmstart": drawdown_selection.get("decision", ""),
        "release_ppo": ppo_selection.get("decision", ""),
        "drawdown_ppo": dd_ppo_selection.get("decision", ""),
    },
}

joint_targets.to_csv(OUT_DIR / "joint_target_contract.csv", index=False)
joint_summary.to_csv(OUT_DIR / "joint_target_summary.csv", index=False)
scope_audit.to_csv(OUT_DIR / "target_scope_audit.csv", index=False)
lift_audit.to_csv(OUT_DIR / "target_lift_audit.csv", index=False)
action_distribution.to_csv(OUT_DIR / "action_distribution.csv", index=False)
bc_readiness_scores.to_csv(OUT_DIR / "bc_readiness_scores.csv", index=False)
anti_overfit_stop_rules.to_csv(OUT_DIR / "anti_overfit_stop_rules.csv", index=False)
(OUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
selection


{'experiment': 'cap10_foldlocal_joint_target_contract_v1',
 'created_at': '2026-05-25T22:58:02.180749+00:00',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'cap10_foldlocal_multi_objective_readiness_v1',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'training_objective', 'src_helper'],
 'decision': 'foldlocal_joint_target_contract_ready_for_bc_screen_no_promotion',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'bc_screen_allowed_next': True,
 'target_gate': True,
 'target_scope_gate': True,
 'eval_overlap_rows': 0,
 'outside_train_eval_rows': 0,
 'target_lift_gate': True,
 'current_positive_rebound_lift_rows': 80,
 'candidate_release_positive_lift_rows': 114,
 'rebound_downtrend_rows': 114,
 'bc_readiness_gate': True,
 'selected_samples': 1261,
 'selected_unique_actions': 18,
 'selected_dominant_action_share': 0.2212529738302934,
 'folds': ['wf_2022_train', 'wf_2023_2024_train'],
 'wf_2020_coverage

In [6]:

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`.

The new joint target contract is ready for a **BC screen only**, not promotion. It contains `{len(joint_targets)}` fold-local rows across `{', '.join(selection['folds'])}` with `eval_overlap_rows={selection['eval_overlap_rows']}` and `outside_train_eval_rows={selection['outside_train_eval_rows']}`.

Key target gates:
- Scope gate: `{selection['target_scope_gate']}`.
- Lift gate: `{selection['target_lift_gate']}` with `{selection['current_positive_rebound_lift_rows']}` current positive rebound-lift rows and `{selection['rebound_downtrend_rows']}` rebound/downtrend rows.
- BC readiness gate: `{selection['bc_readiness_gate']}` with `{selection['selected_samples']}` samples, `{selection['selected_unique_actions']}` unique actions, and dominant action share `{selection['selected_dominant_action_share']:.3f}`.

Risk notes:
- This is not a backtest and cannot support runtime promotion.
- Previous PPO screens showed drawdown drift, so the next BC/PPO notebook must enforce `anti_overfit_stop_rules.csv`.
- `wf_2020` coverage is still absent from this target contract and remains a promotion blocker until a non-leaking replay/coverage path is added.

Next action: run a BC-only slice screen from `joint_target_contract.csv`, then allow PPO only if the BC release/downtrend slices pass and checkpoint selection rejects drawdown drift.
"""
display(Markdown(conclusion))



## Conclusion

Decision: `foldlocal_joint_target_contract_ready_for_bc_screen_no_promotion`.

The new joint target contract is ready for a **BC screen only**, not promotion. It contains `1261` fold-local rows across `wf_2022_train, wf_2023_2024_train` with `eval_overlap_rows=0` and `outside_train_eval_rows=0`.

Key target gates:
- Scope gate: `True`.
- Lift gate: `True` with `80` current positive rebound-lift rows and `114` rebound/downtrend rows.
- BC readiness gate: `True` with `1261` samples, `18` unique actions, and dominant action share `0.221`.

Risk notes:
- This is not a backtest and cannot support runtime promotion.
- Previous PPO screens showed drawdown drift, so the next BC/PPO notebook must enforce `anti_overfit_stop_rules.csv`.
- `wf_2020` coverage is still absent from this target contract and remains a promotion blocker until a non-leaking replay/coverage path is added.

Next action: run a BC-only slice screen from `joint_target_contract.csv`, then allow PPO only if the BC release/downtrend slices pass and checkpoint selection rejects drawdown drift.
